# Employee ETL Pipeline Walkthrough
This notebook demonstrates the Extract, Transform, Load (ETL) pipeline step-by-step.

In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
import sys

# Add parent directory to path to import config
sys.path.append(os.path.abspath('..'))
import config

## Step 1: Extract Data
Load the raw data from the CSV file and inspect it.

In [ ]:
data_path = '../data/employees.csv'
df_raw = pd.read_csv(data_path)
print(f"Raw data shape: {df_raw.shape}")
display(df_raw.head())

## Step 2: Transform Data
Clean the data: standardizing columns, removing duplicates, handling missing values, converting data types, and filtering invalid records.

In [ ]:
df_clean = df_raw.copy()

# 1. Standardize columns
df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')

# 2. Drop duplicates
df_clean = df_clean.drop_duplicates()

# 3. Handle nulls
df_clean['department'] = df_clean.apply(lambda row: 'Unknown' if pd.isna(row['department']) else row['department'], axis=1)
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')
median_age = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(median_age).astype(int)

# 4. Fix salary
df_clean['salary'] = pd.to_numeric(df_clean['salary'], errors='coerce')
df_clean = df_clean.dropna(subset=['salary'])
df_clean = df_clean[df_clean['salary'] > 0]
df_clean['salary'] = df_clean['salary'].astype(int)

# 5. Fix dates
df_clean['join_date'] = pd.to_datetime(df_clean['join_date'], errors='coerce')
df_clean = df_clean.dropna(subset=['join_date'])

print(f"Cleaned data shape: {df_clean.shape}")
display(df_clean.head())

## Step 3: Load Data
Load the cleaned data into the MySQL database.

In [ ]:
try:
    engine = create_engine(config.DB_URL)
    df_clean.to_sql(name='employees', con=engine, if_exists='replace', index=False)
    print("Data loaded successfully into MySQL!")
except Exception as e:
    print(f"Error loading data: {e}")